# Week 5 Tutorial — Assignment A2, Part B
## Machine Learning: Model comparison
**In-class activity:** An Ergonomic Assessment Tool With a Smartphone in Pocket

**Part B of A2:** Train KNN, SVM and Random Forest on the Part A pipeline, and compare
all four models.

> **This is the second half of the same assignment you started last week**, not a new
> one. Part A was worth **55 marks** and Part B is worth the remaining **45**, marked
> together as one piece of work out of 100.
>
> Last week's upload was a progress record and was not graded. **This week's is the real
> one.** Continue answering in the same question document, and submit the completed
> version — Parts A and B together, in one file — at the end of this tutorial. If your
> submission contains only Part B, the 55 marks for Part A cannot be awarded.

### Objective

Last week you built the full pipeline and measured a Logistic Regression baseline. This
week the pipeline is unchanged — **same features, same 20 RFE selections, same
stratified split, same `random_state`** — and you add three models that work on
completely different principles:

| Model | How it decides | What it can represent that LR cannot |
|---|---|---|
| Logistic Regression *(Part A baseline)* | one linear boundary per class | — |
| K-Nearest Neighbours | the majority label among the 5 closest training windows | local, irregularly shaped regions |
| Support Vector Machine (RBF) | a boundary in a transformed space | smooth curved boundaries |
| Random Forest | a vote across 300 decision trees | feature interactions and thresholds |

Holding the pipeline fixed is what makes the comparison honest. If the models saw
different features or different splits, a difference in macro F1 would tell you nothing
about the models.

### Steps
1. Setup and rebuild the Part A pipeline
2. Retrain the Logistic Regression baseline (reference bar)
3. Train KNN
4. Train SVM and Random Forest
5. Final four-model test comparison
6. Improvement proposal

## Learning outcomes

By the end of this in-class assignment, you should be able to:

1. Rebuild a fixed preprocessing and feature-selection pipeline so that several models
   are compared on identical inputs.
2. Build KNN, SVM and Random Forest classifiers, and state the key settings of each.
3. Explain which models need feature scaling and which do not, and why.
4. Compare four models using Accuracy, macro Precision, macro Recall, macro F1-score and
   confusion matrices.
5. Select a model using macro F1-score rather than Accuracy, and justify the choice
   against a baseline.
6. Propose a concrete improvement based on the per-class errors you observe.

> **Important:** A2 is an **in-class assignment**. Complete this notebook, ensure all
> required outputs are visible, and export the **complete** question document — Parts A
> and B — as a PDF before leaving.

## 60-minute flow — Part B, questions 7 to 11

| Time | Activity | Student output | Q | Marks |
|---|---|---|---:|---:|
| 0-5 min | Recap: the Part A baseline and why it matters | Part A baseline test macro F1 restated | - | - |
| 5-15 min | Rebuild the pipeline and retrain the baseline | Split sizes and baseline validation metrics reproduced | 7 | 5 |
| 15-27 min | Train KNN | Standardized KNN pipeline, validation metrics, confusion matrix | 8 | 8 |
| 27-42 min | Train SVM and Random Forest | Both models trained, validation results, key settings stated | 9 | 12 |
| 42-54 min | Final four-model test comparison | Comparison table, comparison plot, best model justified on macro F1 | 10 | 13 |
| 54-58 min | Improvement proposal | One concrete, justified improvement | 11 | 7 |
| 58-60 min | Submit the complete assignment | Exported PDF, Parts A and B, with visible code, outputs, figures and written answers | - | - |
|  | **Part B subtotal** |  |  | **45** |
|  | *Part A (Week 4), questions 1-6* |  |  | *55* |
|  | **Assignment total** |  |  | **100** |

## Step 1: Rebuild the Part A pipeline

The cells in this section are **identical to Part A**. Run them in order to get back to
the same feature matrix, the same 20 selected features, and the same three splits.

Nothing here is new work. Do not change any `random_state` value — they are what make
your numbers reproducible and comparable both with Part A and with your classmates'.

Step 1: Setup environment

In [ ]:
import math
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from google.colab import files
import io

In [ ]:
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['font.size'] = 14
plt.rcParams['font.serif'] = 'Times New Roman'
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd

github_csv_url = "https://raw.githubusercontent.com/yantaolab/Autonomous-Construction-and-Robotics/main/Projects/Project%2002%20Smartphone-based%20pose%20classification/preprocessed%20dataset%20823.csv"

try:
    df = pd.read_csv(github_csv_url)
except Exception as e:
    try:
        df = pd.read_csv(github_csv_url, sep='\t')
    except Exception:
        raise RuntimeError(f"Failed to load dataset from github url! error:{e}")
print(f"Loaded dataset from GitHub.")

In [ ]:
smoothed_data = df
smoothed_data

In [ ]:
smoothed_data = smoothed_data[smoothed_data['Label_2'].isin([1,2,5,8])]

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, entropy
from scipy.fftpack import fft

# Calculate magnitude (combines multi-axis sensor data into a single scalar to represent overall signal strength),
# This step consolidates 3D axis data into a 1D magnitude value, eliminating directionality to focus on the overall intensity of the sensor signal (e.g., total acceleration).

def compute_magnitude(df, cols):
    return np.sqrt((df[cols] ** 2).sum(axis=1))

# Calculate jerk (first derivative of sensor signals, representing the rate of change of motion)
# Jerk reflects how quickly motion changes (e.g., sudden starts/stops), which is a key feature for distinguishing motion patterns (e.g., walking vs. bending).
def compute_jerk(df, cols):
    return df[cols].diff().fillna(0)

# define features statistic
def extract_stat_features(segment):
    return {
        'mean': segment.mean(),
        'std': segment.std(),
        'min': segment.min(),
        'max': segment.max(),
        'skew': skew(segment),
        'kurtosis': kurtosis(segment)
    }

# Extract freqency features
def extract_freq_features(signal):
    fft_vals = np.abs(fft(signal.values))[:len(signal)//2]  #
    energy = np.sum(fft_vals ** 2)
    prob = fft_vals / np.sum(fft_vals)
    spec_entropy = entropy(prob)
    return {
        'fft_energy': energy,
        'fft_entropy': spec_entropy
    }


# Pitch and Roll
def extract_posture_angles(df):
    g = df[['Gravity_x', 'Gravity_y', 'Gravity_z']].values
    pitch = np.arctan2(g[:, 0], np.sqrt(g[:, 1] ** 2 + g[:, 2] ** 2))
    roll = np.arctan2(g[:, 1], g[:, 2])
    return pd.Series({
        'pitch_mean': pitch.mean(),
        'roll_mean': roll.mean(),
        'pitch_std': pitch.std(),
        'roll_std': roll.std()
    })


In [ ]:
def extract_all_features(data, window_size, step):
    features = []
    for start in range(0, len(data) - window_size, step):
        end = start + window_size
        segment = data.iloc[start:end]

        f = {}

        # original magnitude
        f['acc_mag_mean'] = compute_magnitude(segment, ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z']).mean()
        f['gyro_mag_mean'] = compute_magnitude(segment, ['Gyroscope_x','Gyroscope_y','Gyroscope_z']).mean()

        # jerk and jerk magnitude
        jerk = compute_jerk(segment, ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z'])
        jerk_mag = compute_magnitude(jerk, ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z'])
        f['jerk_mag_mean'] = jerk_mag.mean()
        f['jerk_mag_std'] = jerk_mag.std()

        # Extract stat and freq features from accelerometer and gyroscope
        for col in ['Accelerometer.xlsx_x','Accelerometer.xlsx_y','Accelerometer.xlsx_z',
                    'Gyroscope_x','Gyroscope_y','Gyroscope_z']:
            stats = extract_stat_features(segment[col])
            freq = extract_freq_features(segment[col])
            for k, v in stats.items():
                f[f'{col}_{k}'] = v
            for k, v in freq.items():
                f[f'{col}_{k}'] = v

        # Posture angle extraction
        posture_angles = extract_posture_angles(segment)
        f.update(posture_angles)

        # Correspondence calculation
        f['acc_xy_corr'] = segment['Accelerometer.xlsx_x'].corr(segment['Accelerometer.xlsx_y'])
        f['gyro_xy_corr'] = segment['Gyroscope_x'].corr(segment['Gyroscope_y'])

        # Majority vote
        f['label'] = segment['Label_2'].mode()[0]

        features.append(f)
    return pd.DataFrame(features)


In [ ]:
feature_df = extract_all_features(smoothed_data, window_size=5, step=10)

In [ ]:
feature_df

In [ ]:
# label reflection
label_mapping = {1: 0, 2: 1, 5: 2, 8: 3}
feature_df['label'] = feature_df['label'].map(label_mapping)

In [ ]:
# Define all engineered features and the target label
X_all = feature_df.drop(columns=["label"])
y_all = feature_df["label"]

print("Feature matrix shape:", X_all.shape)
print("Target shape:", y_all.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% development data and 20% final test data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=2,
    stratify=y_all
)

# Second split: 75% train and 25% validation within development data
# Overall ratio: 60% train, 20% validation, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=2,
    stratify=y_train_val
)

print(f"Train set:      {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

# Fit feature selection using training data only.
rfe_estimator = RandomForestClassifier(
    n_estimators=100,
    random_state=2,
    n_jobs=-1
)

rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=20,
    step=1
)

rfe.fit(X_train, y_train)

selected_features = X_train.columns[rfe.support_].tolist()

feature_ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "Rank": rfe.ranking_,
    "Selected": rfe.support_
}).sort_values(["Selected", "Rank"], ascending=[False, True])

print("Top 20 selected features:")
print(selected_features)
display(feature_ranking.head(20))

In [ ]:
# Retain the same selected features in every dataset split
train_X = X_train[selected_features].copy()
val_X = X_val[selected_features].copy()
test_X = X_test[selected_features].copy()

train_Y = y_train.copy()
val_Y = y_val.copy()
test_Y = y_test.copy()

print("Selected-feature training shape:", train_X.shape)
print("Selected-feature validation shape:", val_X.shape)
print("Selected-feature test shape:", test_X.shape)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    metrics = {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1-score (macro)": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )
    }

    print(f"===== {model_name}: {dataset_name} set =====")
    for metric, value in list(metrics.items())[2:]:
        print(f"{metric}: {value:.4f}")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_true, y_pred),
        display_labels=["Standing", "Walking", "Bending", "Squatting"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"{model_name}: {dataset_name} confusion matrix")
    plt.tight_layout()
    plt.show()

    return metrics

## Step 2: Retrain the baseline — Question 7, 5 points

Refit the Part A Logistic Regression model so its scores sit in the same table as the
three new models.

### Your task

1. Run the cell below.
2. Confirm the validation macro F1-score matches what you recorded in Question 5. If it
   does not, something in the pipeline above was changed — find it before continuing.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Unchanged from Part A
logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=2000,
        C=1.0,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42,
        n_jobs=-1
    ))
])

logreg_model.fit(train_X, train_Y)

logreg_val_pred = logreg_model.predict(val_X)
logreg_test_pred = logreg_model.predict(test_X)

logreg_val_metrics = evaluate_model(
    val_Y, logreg_val_pred, "Logistic Regression", "Validation"
)

## Step 3: KNN — Question 8, 8 points

K-Nearest Neighbours stores the training set and classifies a new window by the majority
label among its 5 nearest training windows. There is no fitted equation at all.

It sits in a scaling pipeline because it is **distance-based**: without scaling, a
feature measured in hundreds would dominate the distance calculation regardless of how
informative it is. This is the same reason LR was scaled, but the consequence is more
severe — for LR, scaling affects the penalty; for KNN, it changes which points count as
"near".

### Your task

1. Train KNN on the training set.
2. Evaluate on the validation set and report all four metrics plus the confusion matrix.
3. State the key settings (`n_neighbors`, `weights`, distance metric).
4. In 1-2 sentences, compare its validation macro F1 with the baseline. Did the extra
   flexibility help?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        metric="minkowski",
        p=2,
        n_jobs=-1
    ))
])

# Train
knn_model.fit(train_X, train_Y)

# Predictions
knn_val_pred = knn_model.predict(val_X)
knn_test_pred = knn_model.predict(test_X)
knn_test_prob = knn_model.predict_proba(test_X)

# Evaluation

knn_val_metrics = evaluate_model(
    val_Y, knn_val_pred, "KNN", "Validation"
)

## Step 4: SVM and Random Forest — Question 9, 12 points

**SVM with an RBF kernel** finds a boundary that can curve, by implicitly mapping the
features into a higher-dimensional space. `C=10` controls how much it is willing to
misclassify training points to keep the boundary smooth; a high `C` fits training data
more tightly. It is distance-based, so it is scaled.

**Random Forest** averages 300 decision trees, each grown on a bootstrap sample with a
random feature subset per split. It is the only model here that does **not** need
scaling: a tree splits one feature at a time on a threshold, and a threshold is
unaffected by the units of any other feature.

Both use `class_weight="balanced"` for the same reason as the baseline — bending is 6% of
the data.

### Your task

1. Train the SVM inside a standardized pipeline.
2. Train the Random Forest **without** a scaler, and be able to say why that is correct.
3. Evaluate both on the validation set.
4. State the key settings for each model.
5. Briefly compare the validation macro F1-scores of all four models so far.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=42
    ))
])

# Train
svm_model.fit(train_X, train_Y)

# Predictions
svm_val_pred = svm_model.predict(val_X)
svm_test_pred = svm_model.predict(test_X)
svm_test_prob = svm_model.predict_proba(test_X)

# Evaluation
svm_val_metrics = evaluate_model(
    val_Y, svm_val_pred, "SVM", "Validation"
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(train_X, train_Y)

# Predictions
rf_val_pred = rf_model.predict(val_X)
rf_test_pred = rf_model.predict(test_X)
rf_test_prob = rf_model.predict_proba(test_X)

# Evaluation
rf_val_metrics = evaluate_model(
    val_Y, rf_val_pred, "Random Forest", "Validation"
)

## Step 5: Final four-model test comparison — Question 10, 13 points

Now evaluate all four final models once on the untouched test set.

### Your task

1. Create a comparison table with Accuracy, macro Precision, macro Recall and macro
   F1-score for Logistic Regression, KNN, SVM and Random Forest.
2. Create the comparison plot.
3. Select the best model using **macro F1-score** as the primary criterion.
4. In 2-3 sentences, justify the choice using the test metrics and the confusion matrix.
   State how much the best model gains over the Logistic Regression baseline.
5. Answer one further question in 1-2 sentences: is the gain over the baseline large
   enough to justify the more complex model? A model that is harder to explain and
   slower to run has to earn that cost.

Do not select a model based only on Accuracy. With squatting at roughly half the data, a
model that never predicts bending still scores highly on Accuracy. Macro F1-score gives
equal importance to each posture class, which is what an ergonomic-risk tool needs —
bending and squatting are the postures the tool exists to detect.

In [ ]:
# Final predictions on the untouched test set
logreg_test_pred = logreg_model.predict(test_X)
knn_test_pred = knn_model.predict(test_X)
svm_test_pred = svm_model.predict(test_X)
rf_test_pred = rf_model.predict(test_X)

# Final metrics
logreg_test_metrics = evaluate_model(
    test_Y, logreg_test_pred, "Logistic Regression", "Test"
)

knn_test_metrics = evaluate_model(
    test_Y, knn_test_pred, "KNN", "Test"
)

svm_test_metrics = evaluate_model(
    test_Y, svm_test_pred, "SVM", "Test"
)

rf_test_metrics = evaluate_model(
    test_Y, rf_test_pred, "Random Forest", "Test"
)

comparison_df = pd.DataFrame([
    logreg_test_metrics,
    knn_test_metrics,
    svm_test_metrics,
    rf_test_metrics
]).drop(columns="Dataset")

comparison_df = comparison_df.sort_values(
    "F1-score (macro)",
    ascending=False
).reset_index(drop=True)

display(comparison_df.round(4))

comparison_long = comparison_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Score",
    hue="Model",
    palette="Set2"
)

# The baseline drawn as a line: every bar above it is a real gain
baseline_f1 = pd.DataFrame([logreg_test_metrics])["F1-score (macro)"].iloc[0]
plt.axhline(
    baseline_f1,
    color="#6B7480",
    linestyle="--",
    linewidth=2,
    label=f"LR baseline macro F1 = {baseline_f1:.3f}"
)

plt.ylim(0, 1.05)
plt.title("Final Test-set Performance: four models against the baseline")
plt.xlabel("")
plt.ylabel("Score")
plt.legend(loc="lower right", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

best_model = comparison_df.loc[0, "Model"]
best_f1 = comparison_df.loc[0, "F1-score (macro)"]

print(f"Best model by test macro F1-score: {best_model}")
print(f"Best macro F1-score: {best_f1:.4f}")
print(f"Logistic Regression baseline: {baseline_f1:.4f}")
print(f"Gain over baseline: {best_f1 - baseline_f1:+.4f}")

## Step 6: Improvement proposal — Question 11, 7 points

Look at the confusion matrix of your best model and find its largest off-diagonal cell —
the pair of postures it confuses most often.

### Your task

Propose **one** concrete improvement, in 3-5 sentences. State:

1. Which specific error you are targeting (name the two postures and cite the count).
2. What you would change — a feature, the window length, the sampling of a class, the
   model, or the labelling.
3. Why that change should address *that specific error*, not performance in general.

Some directions worth considering:

- **Window length.** The features use `window_size=5` with `step=10`. At the sampling
  rate of this dataset that is a very short window, and windows do not overlap. A longer
  window captures a fuller movement cycle but blurs transitions between postures.
- **The majority-vote label.** Each window is labelled by the most common label inside
  it, so a window spanning a posture change gets one label for mixed content.
- **Class imbalance.** `class_weight="balanced"` reweights the loss, but it does not
  create more bending examples. Resampling is another option — and it must be fitted on
  the training set only.
- **Features that are missing.** Nothing in the current feature set describes how a
  posture *is entered*. Bending and squatting both end with the torso lowered.
- **One worker.** All of this data comes from a single scaffolder in one session. Say
  what that means for a model deployed to a different worker.

A good answer names the error and the mechanism. "Use a deep learning model" is not an
answer.

---

## Before you submit

This is the **marked** submission: the complete A2 document, Parts A and B together.

- [ ] Part A (questions 1-6) is still complete and present in the document.
- [ ] Split sizes reproduced and the 20 selected features listed.
- [ ] Baseline validation metrics matching your Part A record.
- [ ] Validation metrics and confusion matrices for KNN, SVM and Random Forest.
- [ ] The four-model test comparison table.
- [ ] The comparison plot, with the baseline line visible.
- [ ] Best model stated and justified on macro F1-score, with the gain over the baseline.
- [ ] Key settings stated for each of the four models.
- [ ] The improvement proposal, naming a specific confusion and a mechanism.

**Filename:** `A2_<YourName>.pdf` — one file, 100 marks.